# Resume Training YOLO11n (Kaggle)
Notebook ini dikhususkan untuk **MELANJUTKAN** sesi training Kaggle yang sebelumnya terputus (misalnya terkena batas 12 jam).

**Syarat Wajib Sebelum Menekan Run All:**
1. Klik tombol **Add Data** di kanan atas Kaggle.
2. Pilih tab **Your Work** (atau Notebooks).
3. Tambahkan Notebook Kaggle Anda yang sebelumnya.
4. Ubah variabel `INPUT_DATASET_NAME` di bawah agar sesuai dengan nama folder yang baru Anda tambahkan.

In [ ]:
import os
import shutil

print("🔍 Memindai seluruh folder /kaggle/input/ secara otomatis...")

found_path = None
# Berjalan menelusuri setiap lorong folder di dalam input Kaggle
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        found_path = root
        break

if found_path:
    print(f"🎯 BINGO! Dataset asli Anda ngumpet di: {found_path}")
    
    dataset_dest = '/kaggle/working/vnetra_master_dataset'
    
    # Hapus jika sudah ada sisa sebelumnya
    if os.path.exists(dataset_dest):
        shutil.rmtree(dataset_dest)
        
    print("⏳ Menyalin ke folder kerja (tunggu beberapa detik)...")
    shutil.copytree(found_path, dataset_dest)
    print("✅ Salin Dataset Selesai! Anda siap lanjut ke cell berikutnya.")
else:
    print("❌ Waduh, file data.yaml sama sekali tidak ditemukan di dalam /kaggle/input/!")
    print("Pastikan Anda sudah menekan 'Add Data' dan datasetnya benar-benar berisi data.yaml.")



## 1. Menyalin Dataset & Kapsul Waktu Secara Internal
Alih-alih mengunduh ulang 120.000 gambar dari Roboflow, kita cukup menyalin hasil dataset matang dari sesi kemarin secara internal di server Kaggle. Ini hanya memakan waktu beberapa detik!

In [ ]:
from ultralytics import YOLO
import os

checkpoint_file = 'epoch80.pt'
checkpoint_path = f'/kaggle/input/datasets/aanandiyanasandi/vnetra-master-dataset/{checkpoint_file}'

if not os.path.exists(checkpoint_path):
    print("❌ ERROR: Kapsul Waktu tidak ditemukan!")
else:
    print("✅ Kapsul Waktu ditemukan! Membangunkan model YOLO...")
    model = YOLO(checkpoint_path)
    
    print("🚀 Melanjutkan training lintas-platform dengan konfigurasi asli...")
    
    # Mematikan resume=True karena beda struktur directory (Colab vs Kaggle)
    # Melanjutkan sisa epoch dari bobot epoch 80 yang cerdas
    results = model.train(
        data='/kaggle/working/vnetra_master_dataset/data.yaml', 
        epochs=220,           # Sisa target epoch (300 - 80)
        imgsz=640,            # SAMAKAN dengan notebook awal
        batch=100,            # SAMAKAN dengan notebook awal
        project='vnetra_training',
        name='yolo11n_custom'
    )



## 2. Melanjutkan (Resume) Training
Kita akan membangunkan model dari kapsul waktu `last.pt` milik sesi kemarin.

In [ ]:
from ultralytics import YOLO
import os

# Jika last.pt rusak (0 bytes), Anda bisa menggantinya dengan 'epoch80.pt' atau checkpoint lain yang selamat.
checkpoint_file = 'last.pt'
checkpoint_path = f'/kaggle/input/{INPUT_DATASET_NAME}/vnetra_training/yolo11n_custom/weights/{checkpoint_file}'

print(f"🔍 Memeriksa kapsul waktu di: {checkpoint_path}...")

if not os.path.exists(checkpoint_path):
    print("❌ ERROR: Kapsul Waktu (Checkpoint) tidak ditemukan!")
else:
    print("✅ Kapsul Waktu ditemukan! Membangunkan model YOLO...")
    model = YOLO(checkpoint_path)
    
    # --- [OPSIONAL] ALARM REM DARURAT SISA KUOTA ---
    # Jika sisa kuota Kaggle Anda tinggal sedikit (misal 2 Jam), aktifkan blok kode di bawah ini
    # dengan menghapus tanda pagar (#) di awal baris. YOLO akan berhenti dengan aman di jam ke-1.5.
    # 
    # import time
    # def alarm_kuota(trainer):
    #     if time.time() - trainer.train_time_start > 5400:  # 5400 detik = 1.5 jam
    #         print("🚨 ALARM: Sisa kuota hampir habis! Menyimpan progress...")
    #         trainer.stop = True
    # model.add_callback("on_train_epoch_end", alarm_kuota)
    # -----------------------------------------------
    
    print("🚀 Melanjutkan training dari titik terakhir...")
    results = model.train(resume=True)


## 5. Export ke TensorFlow Lite (TFLite)
Mengekspor bobot model menjadi format `.tflite` dalam dua bentuk kuantisasi:
1. **FP16** (Half Precision) -> Sangat efisien dan kompatibel untuk *GPU Delegation* di Android.
2. **INT8** (Full Integer) -> Wajib untuk akselerator *NPU / NNAPI* yang membutuhkan model super ringan.

In [ ]:
# [PATCH] Downgrade protobuf untuk mencegah error 'MessageFactory' object has no attribute 'GetPrototype'
!pip install "protobuf<=3.20.3" -q

print("Mengekspor model ke FP16...")
# 1. Export ke TFLite (FP16) - Optimal untuk GPU Mobile (Proses cepat)
export_fp16 = model.export(format="tflite", half=True, optimize=True)
print("===========================================================")
print("Export FP16 Selesai! Lokasi file TFLite:")
print("FP16:", export_fp16)
print("===========================================================")


In [ ]:
import shutil
import os
shutil.copy(export_fp16, '/kaggle/working/best_fp16.tflite')
print('Model FP16 siap di-download!')

## 6. Validasi Kuantisasi (Benchmarking Skripsi)
Menguji kembali model pada Test Set untuk melihat seberapa jauh penurunan akurasi (mAP) akibat proses kompresi FP16 dibanding model aslinya.

In [ ]:
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) PADA TEST SET ===")
val_pt = model.val(data=f"{master_dir}/data.yaml", split='test')
map_pt = val_pt.box.map50

print("\n=========================================")
print("mAP@50 (Akurasi):")
print(f"Original (.pt)   : {map_pt:.4f}")

In [ ]:
print("\n=== EVALUASI MODEL FP16 (.tflite) PADA TEST SET ===")
model_fp16 = YOLO(export_fp16, task='detect')
val_fp16 = model_fp16.val(data=f"{master_dir}/data.yaml", split='test')
map_fp16 = val_fp16.box.map50

print("\n=========================================")
print("mAP@50 (Akurasi):")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")

In [ ]:
print("\n=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 PADA DATASET TEST:")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")
print("=========================================")

## 7. Pengujian Visualisasi Langsung (Predict)
Mengambil satu gambar tes secara acak dan menampilkan prediksi kotak deteksi dari model asli (.pt) vs model terkompresi (.tflite) agar Anda bisa meletakkannya di Laporan Skripsi.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import glob

# Pilih satu gambar acak dari dataset test
test_images = glob.glob(f"{master_dir}/test/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    print(f"Menguji gambar: {test_img}")
    
    # Prediksi pakai model Asli
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    # Prediksi pakai model FP16
    res_fp16 = model_fp16.predict(source=test_img, imgsz=640)
    img_fp16 = res_fp16[0].plot()
    
    # Tampilkan perbandingan
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli (.pt)")
    ax[0].axis("off")
    
    ax[1].imshow(cv2.cvtColor(img_fp16, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Model FP16 (.tflite)")
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada gambar di folder test untuk diprediksi.")


## 8. Visualisasi Grafik Hasil Training
Menampilkan grafik metrik akurasi (*mAP*, *Loss*) dan *Confusion Matrix* yang telah digenerasi oleh YOLO menggunakan `matplotlib` untuk keperluan laporan skripsi.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

base_path = '/kaggle/working/runs/detect/vnetra_training/yolo11n_custom/'
results_path = os.path.join(base_path, 'results.csv')

print('=== 1. CUSTOM TRAINING DASHBOARD (SEABORN) ===')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()  # Bersihkan spasi berlebih di nama kolom
    
    sns.set_theme(style='whitegrid', palette='deep')
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('VNetra - YOLO11n Training Performance Dashboard', fontsize=26, fontweight='bold', y=0.96)
    
    # 1. Box Loss Convergence
    sns.lineplot(data=df, x='epoch', y='train/box_loss', ax=axes[0,0], label='Train Box Loss', linewidth=3, color='#1f77b4')
    sns.lineplot(data=df, x='epoch', y='val/box_loss', ax=axes[0,0], label='Val Box Loss', linewidth=3, color='#ff7f0e', linestyle='--')
    axes[0,0].set_title('Box Loss Convergence', fontsize=18, fontweight='bold')
    axes[0,0].set_xlabel('Epoch', fontsize=14)
    axes[0,0].set_ylabel('Loss', fontsize=14)
    axes[0,0].legend(fontsize=12, frameon=True, shadow=True)
    
    # 2. Class Loss Convergence
    sns.lineplot(data=df, x='epoch', y='train/cls_loss', ax=axes[0,1], label='Train Class Loss', linewidth=3, color='#2ca02c')
    sns.lineplot(data=df, x='epoch', y='val/cls_loss', ax=axes[0,1], label='Val Class Loss', linewidth=3, color='#d62728', linestyle='--')
    axes[0,1].set_title('Classification Loss Convergence', fontsize=18, fontweight='bold')
    axes[0,1].set_xlabel('Epoch', fontsize=14)
    axes[0,1].set_ylabel('Loss', fontsize=14)
    axes[0,1].legend(fontsize=12, frameon=True, shadow=True)
    
    # 3. mAP Score Evolution
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50(B)', ax=axes[1,0], label='mAP@50', linewidth=3, color='#9467bd')
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50-95(B)', ax=axes[1,0], label='mAP@50-95', linewidth=3, color='#8c564b', linestyle='-.')
    axes[1,0].set_title('Mean Average Precision (mAP)', fontsize=18, fontweight='bold')
    axes[1,0].set_xlabel('Epoch', fontsize=14)
    axes[1,0].set_ylabel('Score', fontsize=14)
    axes[1,0].legend(fontsize=12, frameon=True, shadow=True)
    
    # 4. Precision & Recall
    sns.lineplot(data=df, x='epoch', y='metrics/precision(B)', ax=axes[1,1], label='Precision', linewidth=3, color='#e377c2')
    sns.lineplot(data=df, x='epoch', y='metrics/recall(B)', ax=axes[1,1], label='Recall', linewidth=3, color='#17becf', linestyle=':')
    axes[1,1].set_title('Precision & Recall Trends', fontsize=18, fontweight='bold')
    axes[1,1].set_xlabel('Epoch', fontsize=14)
    axes[1,1].set_ylabel('Score', fontsize=14)
    axes[1,1].legend(fontsize=12, frameon=True, shadow=True)
    
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
else:
    print('File results.csv belum ditemukan.')

def display_result(image_path, width=None):
    if os.path.exists(image_path):
        if width:
            display(Image(filename=image_path, width=width))
        else:
            display(Image(filename=image_path))
    else:
        print(f'\u26a0\ufe0f Gambar tidak ditemukan: {os.path.basename(image_path)}')
        print('   (Gambar ini baru akan digenerate oleh YOLO di detik terakhir setelah epoch 50 selesai 100%)')

print('\n=== 2. CONFUSION MATRIX PROFESIONAL ===')
display_result(os.path.join(base_path, 'confusion_matrix_normalized.png'), width=1200)

print('\n=== 3. KURVA F1-SCORE (Confidence Thresholding) ===')
display_result(os.path.join(base_path, 'F1_curve.png'), width=1200)

print('\n=== 4. VISUALISASI AUGMENTASI MOSAIC PADA DATA TRAINING ===')
display_result(os.path.join(base_path, 'train_batch0.jpg'), width=1200)

print('\n=== 5. SAMPEL PREDIKSI PADA VALIDATION SET ===')
display_result(os.path.join(base_path, 'val_batch0_pred.jpg'), width=1200)



## 9. Paketkan Hasil Training (ZIP)
Membungkus seluruh grafik evaluasi, kurva performa, matriks kebingungan (*confusion matrix*), dan file model bobot (`best.pt` & `best_fp16.tflite`) ke dalam satu file ZIP yang sangat praktis untuk Anda unduh (tanpa mengikutkan dataset gambar untuk menghemat kuota).

In [ ]:
import os
import zipfile

zip_path = "/kaggle/working/vnetra_training_results.zip"
train_dir = "/kaggle/working/runs/detect/vnetra_training/yolo11n_custom"
extra_files = ["/kaggle/working/best_fp16.tflite"]

print("Membuat arsip ZIP untuk hasil training...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Memasukkan seluruh folder hasil training YOLO (Berisi grafik, metrik, best.pt)
    if os.path.exists(train_dir):
        for root, dirs, files in os.walk(train_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Menentukan struktur folder di dalam ZIP
                arcname = os.path.relpath(file_path, "/kaggle/working")
                zipf.write(file_path, arcname)
    else:
        print("Peringatan: Folder training YOLO tidak ditemukan!")
        
    # Memasukkan file satuan tambahan seperti FP16 TFLite
    for f in extra_files:
        if os.path.exists(f):
            arcname = os.path.basename(f)
            zipf.write(f, arcname)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\u2705 Berhasil! Silakan unduh file: {zip_path} ({size_mb:.2f} MB)")
print("File ini berisi semua model terlatih dan grafik untuk skripsi Anda.")